# Operational Neural Network - Operator Analysis

This notebook explores how ONN operators work and evolve during training.
It demonstrates:
- How each operator (polynomial, sinusoidal, Gaussian, multiplicative) transforms inputs
- Operator parameter evolution across training epochs
- Side-by-side comparison of ONN vs MLP decision boundaries
- Why ONNs can outperform MLPs on nonlinear tasks

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

from src.data import load_dataset, get_full_tensors
from src.model.operators import PolynomialOperator, SinusoidalOperator, GaussianOperator, MultiplicativeOperator
from src.model.onn_model import ONNModel
from src.model.baseline_model import BaselineMLP
from src.train import train_model
from src.evaluate import compare_models, predict_grid
from src.utils import set_seed, count_parameters, get_device

set_seed(42)
device = get_device()
print(f'Device: {device}')

## 1. Operator Response Curves
Visualize each operator's output as a function of input.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()
x = torch.linspace(-3, 3, 300).unsqueeze(1)  # (300, 1)
x2 = x.expand(-1, 2)  # for operators needing 2 features

operators = [
    ('Polynomial (deg=3)', PolynomialOperator(2, 8, degree=3)),
    ('Sinusoidal', SinusoidalOperator(2, 8)),
    ('Gaussian (RBF)', GaussianOperator(2, 8)),
    ('Multiplicative', MultiplicativeOperator(2, 8)),
]

for ax, (name, op) in zip(axes, operators):
    with torch.no_grad():
        y = op(x2).numpy()
    for j in range(min(y.shape[1], 6)):
        ax.plot(x.numpy(), y[:, j], alpha=0.75, label=f'n{j}')
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Input')
    ax.set_ylabel('Output')
    ax.axhline(0, color='k', lw=0.5)
    ax.axvline(0, color='k', lw=0.5)
    ax.legend(fontsize=7, ncol=3)

plt.suptitle('Operator Response Curves (random initialization)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/nb_operator_responses.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved')

## 2. Train ONN on Spiral Dataset

In [ ]:
import os
os.makedirs('../outputs', exist_ok=True)

DATASET = 'spiral'
EPOCHS = 60
HIDDEN = [32, 32]

train_loader, test_loader, in_f, n_cls = load_dataset(
    name=DATASET, n_samples=1000, noise=0.15, batch_size=64
)
X_train, X_test, y_train, y_test = get_full_tensors(
    name=DATASET, n_samples=1000, noise=0.15
)
print(f'Train: {len(train_loader.dataset)}  Test: {len(test_loader.dataset)}')

In [ ]:
# Train all ONN operator variants + MLP baseline
results = {}
histories = {}
models = {}

for op_name in ['polynomial', 'sinusoidal', 'gaussian', 'multiplicative']:
    set_seed(42)
    model = ONNModel(in_features=in_f, hidden_dims=HIDDEN, n_classes=n_cls, operator=op_name)
    print(f'Training ONN ({op_name}) - {count_parameters(model):,} params...')
    h = train_model(model, train_loader, test_loader, epochs=EPOCHS, lr=1e-3,
                    snapshot_every=max(1, EPOCHS // 10), verbose=False)
    models[f'ONN-{op_name}'] = model
    histories[f'ONN-{op_name}'] = h
    print(f'  best_val_acc={h["best_val_acc"]:.4f}')

set_seed(42)
mlp = BaselineMLP(in_features=in_f, hidden_dims=HIDDEN, n_classes=n_cls)
print(f'Training Baseline MLP - {count_parameters(mlp):,} params...')
h_mlp = train_model(mlp, train_loader, test_loader, epochs=EPOCHS, lr=1e-3, verbose=False)
models['Baseline MLP'] = mlp
histories['Baseline MLP'] = h_mlp
print(f'  best_val_acc={h_mlp["best_val_acc"]:.4f}')

## 3. Decision Boundaries - All Models

In [ ]:
X_np = X_test.numpy()
y_np = y_test.numpy()
x_min, x_max = X_np[:,0].min()-0.5, X_np[:,0].max()+0.5
y_min, y_max = X_np[:,1].min()-0.5, X_np[:,1].max()+0.5

model_list = list(models.items())
n = len(model_list)
fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
if n == 1: axes = [axes]

for ax, (name, model) in zip(axes, model_list):
    xx, yy, Z = predict_grid(model, (x_min, x_max), (y_min, y_max), resolution=150)
    ax.contourf(xx, yy, Z, levels=50, cmap=plt.cm.RdBu, alpha=0.75, vmin=0, vmax=1)
    ax.contour(xx, yy, Z, levels=[0.5], colors='white', lw=1.5, linestyles='--')
    for cls, col in enumerate(['#e74c3c', '#2980b9']):
        mask = y_np == cls
        ax.scatter(X_np[mask,0], X_np[mask,1], c=col, s=15, alpha=0.7, edgecolors='w', lw=0.3)
    ax.set_title(name, fontsize=10)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

plt.suptitle(f'Decision Boundaries - {DATASET.capitalize()}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/nb_decision_boundaries_all.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Operator Parameter Trajectories

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
op_names = ['polynomial', 'sinusoidal', 'gaussian', 'multiplicative']
colors_palette = sns.color_palette('husl', 6)

for ax, op_name in zip(axes, op_names):
    model = models[f'ONN-{op_name}']
    history = model.param_history
    if not history:
        ax.set_title(f'{op_name}\n(no snapshots)')
        continue
    layer_0_history = [h['layer_0'] for h in history]
    param_names = list(layer_0_history[0].keys())
    x_snaps = list(range(1, len(layer_0_history)+1))
    for pname, color in zip(param_names[:6], colors_palette):
        norms = [layer_0_history[t][pname].norm().item() for t in range(len(layer_0_history))]
        ax.plot(x_snaps, norms, marker='o', ms=3, color=color, label=pname)
    ax.set_title(f'{op_name}', fontsize=10)
    ax.set_xlabel('Snapshot')
    ax.set_ylabel('L2 Norm')
    ax.legend(fontsize=6, ncol=2)
    ax.grid(alpha=0.3)

plt.suptitle('Operator Parameter Norm Trajectories (Layer 0)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/nb_param_trajectories.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Accuracy Comparison

In [ ]:
from src.evaluate import evaluate_model

print(f'{'Model':<25} {'Test Acc':>10} {'Best Val':>10} {'Params':>10}')
print('-' * 58)
for name, model in models.items():
    res = evaluate_model(model, test_loader)
    h = histories[name]
    n_params = count_parameters(model)
    print(f'{name:<25} {res["accuracy"]:>10.4f} {h["best_val_acc"]:>10.4f} {n_params:>10,}')